In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/demand_forecasting.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

split_date = df["Date"].quantile(0.8, interpolation="nearest")
train_df = df[df["Date"] <= split_date]
test_df  = df[df["Date"] > split_date]

print("Train range:", train_df["Date"].min(), "to", train_df["Date"].max())
print("Test range:", test_df["Date"].min(), "to", test_df["Date"].max())
print("Train rows:", len(train_df), "| Test rows:", len(test_df))

Train range: 2022-01-01 00:00:00 to 2023-08-31 00:00:00
Test range: 2023-09-01 00:00:00 to 2024-01-30 00:00:00
Train rows: 60800 | Test rows: 15200


In [3]:
df["Week"] = df["Date"].dt.to_period("W-SUN").dt.start_time


weekly = (
    df.groupby(["Product ID", "Week"])
      .agg(
          Demand=("Demand", "sum"),
          Units_Sold=("Units Sold", "sum"),
          Price=("Price", "mean"),
          Discount=("Discount", "mean"),
          Inventory_Level=("Inventory Level", "sum"),
          Category=("Category", "first"),
          Promotion=("Promotion", "max"),
      )
      .reset_index()
      .sort_values(["Product ID", "Week"])
      .reset_index(drop=True)
)

print(weekly.shape)
weekly.head(10)

(2200, 9)


,Product ID,Week,Demand,Units_Sold,Price,Discount,Inventory_Level,Category,Promotion
0,P0001,2021-12-27,1057,828,39.486000,5.000000,1279,Electronics,0
1,P0001,2022-01-03,4476,3302,38.570000,10.142857,10821,Groceries,1
2,P0001,2022-01-10,4143,3582,37.768000,8.000000,10520,Groceries,1
3,P0001,2022-01-17,3818,3154,37.402286,9.142857,13367,Groceries,1
4,P0001,2022-01-24,3952,3416,36.288000,11.571429,10889,Groceries,1
5,P0001,2022-01-31,4071,3626,37.953714,7.714286,12093,Groceries,1
6,P0001,2022-02-07,3632,3286,39.276571,5.714286,11962,Groceries,0
7,P0001,2022-02-14,4121,3548,36.961714,11.857143,12637,Groceries,1
8,P0001,2022-02-21,3919,3550,37.356000,10.428571,10305,Groceries,1
9,P0001,2022-02-28,5070,4436,37.231714,16.571429,13764,Groceries,1


In [4]:
weekly["Year"] = weekly["Week"].dt.isocalendar().year
weekly["WeekNum"] = weekly["Week"].dt.isocalendar().week

baseline_map = weekly.set_index(["Product ID", "Year", "WeekNum"])["Demand"]

def get_naive_forecast(row):
    prev_year = row["Year"] - 1
    key = (row["Product ID"], prev_year, row["WeekNum"])
    return baseline_map.get(key, np.nan)

weekly["Seasonal_Naive"] = weekly.apply(get_naive_forecast, axis=1)

weekly[["Product ID", "Week", "Demand", "Seasonal_Naive"]].tail(10)

,Product ID,Week,Demand,Seasonal_Naive
2190,P0020,2023-11-27,2763,3183.0
2191,P0020,2023-12-04,3083,3378.0
2192,P0020,2023-12-11,3272,3555.0
2193,P0020,2023-12-18,3813,3591.0
2194,P0020,2023-12-25,2369,3573.0
2195,P0020,2024-01-01,1815,2530.0
2196,P0020,2024-01-08,2992,3239.0
2197,P0020,2024-01-15,2899,3295.0
2198,P0020,2024-01-22,3216,2961.0
2199,P0020,2024-01-29,809,1912.0


In [5]:
def wape(actual, forecast):
    actual = np.array(actual)
    forecast = np.array(forecast)
    return np.sum(np.abs(actual - forecast)) / np.sum(np.abs(actual)) * 100

test_weekly = weekly[weekly["Week"] > pd.Timestamp("2023-08-31")].copy()
test_weekly = test_weekly.dropna(subset=["Seasonal_Naive"])

baseline_wape = wape(test_weekly["Demand"], test_weekly["Seasonal_Naive"])
print("Baseline (Seasonal-Naive) WAPE on test period:", round(baseline_wape, 2), "%")

Baseline (Seasonal-Naive) WAPE on test period: 18.04 %


In [6]:
weekly = weekly.sort_values(["Product ID", "Week"]).reset_index(drop=True)

for lag in [1, 2, 3, 4]:
    weekly[f"Demand_lag_{lag}"] = weekly.groupby("Product ID")["Demand"].shift(lag)

weekly["Demand_roll_mean_4"] = (
    weekly.groupby("Product ID")["Demand"]
    .shift(1)
    .rolling(4)
    .mean()
    .reset_index(level=0, drop=True)
)

weekly["Demand_roll_std_4"] = (
    weekly.groupby("Product ID")["Demand"]
    .shift(1)
    .rolling(4)
    .std()
    .reset_index(level=0, drop=True)
)

weekly["Month"] = weekly["Week"].dt.month
weekly["WeekOfYear"] = weekly["Week"].dt.isocalendar().week.astype(int)

weekly[["Product ID", "Week", "Demand", "Demand_lag_1", "Demand_lag_2", "Demand_roll_mean_4"]].tail(10)

,Product ID,Week,Demand,Demand_lag_1,Demand_lag_2,Demand_roll_mean_4
2190,P0020,2023-11-27,2763,2465.0,2994.0,2832.00
2191,P0020,2023-12-04,3083,2763.0,2465.0,2802.75
2192,P0020,2023-12-11,3272,3083.0,2763.0,2826.25
2193,P0020,2023-12-18,3813,3272.0,3083.0,2895.75
2194,P0020,2023-12-25,2369,3813.0,3272.0,3232.75
2195,P0020,2024-01-01,1815,2369.0,3813.0,3134.25
2196,P0020,2024-01-08,2992,1815.0,2369.0,2817.25
2197,P0020,2024-01-15,2899,2992.0,1815.0,2747.25
2198,P0020,2024-01-22,3216,2899.0,2992.0,2518.75
2199,P0020,2024-01-29,809,3216.0,2899.0,2730.50


In [7]:
model_df = weekly.dropna(subset=[
    "Demand_lag_1", "Demand_lag_2", "Demand_lag_3", "Demand_lag_4",
    "Demand_roll_mean_4", "Demand_roll_std_4", "Seasonal_Naive"
]).copy()

feature_cols = [
    "Demand_lag_1", "Demand_lag_2", "Demand_lag_3", "Demand_lag_4",
    "Demand_roll_mean_4", "Demand_roll_std_4",
    "Price", "Discount", "Promotion",
    "Month", "WeekOfYear",
]

target_col = "Demand"

print(model_df.shape)
model_df[feature_cols + [target_col]].isna().sum()

(1160, 20)


Demand_lag_1          0
Demand_lag_2          0
Demand_lag_3          0
Demand_lag_4          0
Demand_roll_mean_4    0
Demand_roll_std_4     0
Price                 0
Discount              0
Promotion             0
Month                 0
WeekOfYear            0
Demand                0
dtype: int64

In [8]:
from xgboost import XGBRegressor

model_df = model_df.sort_values("Week").reset_index(drop=True)
unique_weeks = sorted(model_df["Week"].unique())

n_folds = 4
fold_size = len(unique_weeks) // (n_folds + 1)

fold_results = []

for fold in range(1, n_folds + 1):
    train_end_idx = fold_size * fold
    test_end_idx = fold_size * (fold + 1)

    train_weeks = unique_weeks[:train_end_idx]
    test_weeks = unique_weeks[train_end_idx:test_end_idx]

    train_fold = model_df[model_df["Week"].isin(train_weeks)]
    test_fold = model_df[model_df["Week"].isin(test_weeks)]

    if len(test_fold) == 0:
        continue

    X_train, y_train = train_fold[feature_cols], train_fold[target_col]
    X_test, y_test = test_fold[feature_cols], test_fold[target_col]

    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    fold_wape = wape(y_test, preds)
    baseline_fold_wape = wape(test_fold[target_col], test_fold["Seasonal_Naive"])

    fold_results.append({
        "fold": fold,
        "train_weeks": len(train_weeks),
        "test_weeks": len(test_weeks),
        "model_wape": round(fold_wape, 2),
        "baseline_wape": round(baseline_fold_wape, 2),
    })

fold_results_df = pd.DataFrame(fold_results)
fold_results_df

,fold,train_weeks,test_weeks,model_wape,baseline_wape
0,1,11,11,20.58,18.86
1,2,22,11,10.47,19.20
2,3,33,11,13.14,14.92
3,4,44,11,11.91,16.07


In [9]:
all_preds = []

for fold in range(1, n_folds + 1):
    train_end_idx = fold_size * fold
    test_end_idx = fold_size * (fold + 1)

    train_weeks = unique_weeks[:train_end_idx]
    test_weeks = unique_weeks[train_end_idx:test_end_idx]

    train_fold = model_df[model_df["Week"].isin(train_weeks)]
    test_fold = model_df[model_df["Week"].isin(test_weeks)]

    if len(test_fold) == 0:
        continue

    X_train, y_train = train_fold[feature_cols], train_fold[target_col]
    X_test, y_test = test_fold[feature_cols], test_fold[target_col]

    model = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    temp = test_fold[["Product ID", "Week", "Demand", "Seasonal_Naive"]].copy()
    temp["Model_Pred"] = preds
    temp["fold"] = fold
    all_preds.append(temp)

all_preds_df = pd.concat(all_preds, ignore_index=True)

overall_model_wape = wape(all_preds_df["Demand"], all_preds_df["Model_Pred"])
overall_baseline_wape = wape(all_preds_df["Demand"], all_preds_df["Seasonal_Naive"])

print("Overall Model WAPE:", round(overall_model_wape, 2), "%")
print("Overall Baseline WAPE:", round(overall_baseline_wape, 2), "%")

Overall Model WAPE: 13.77 %
Overall Baseline WAPE: 17.32 %


In [10]:
final_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
)
final_model.fit(model_df[feature_cols], model_df[target_col])

import os
os.makedirs("../models", exist_ok=True)

import pickle
with open("../models/demand_forecast_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open("../models/feature_cols.pkl", "wb") as f:
    pickle.dump(feature_cols, f)

print("Model saved.")

Model saved.


In [11]:
latest_week = model_df["Week"].max()
latest_snapshot = model_df[model_df["Week"] == latest_week].copy()

latest_snapshot["Forecast_Next_4wk"] = final_model.predict(latest_snapshot[feature_cols]) * 4

lead_time_weeks = 2
safety_stock_factor = 1.2

latest_snapshot["Demand_over_leadtime"] = (
    latest_snapshot["Forecast_Next_4wk"] / 4 * lead_time_weeks * safety_stock_factor
)

latest_snapshot["Stockout_Risk"] = (
    latest_snapshot["Inventory_Level"] < latest_snapshot["Demand_over_leadtime"]
).astype(int)

overstock_threshold_weeks = 8
latest_snapshot["Overstock_Risk"] = (
    latest_snapshot["Inventory_Level"] > (latest_snapshot["Forecast_Next_4wk"] / 4 * overstock_threshold_weeks)
).astype(int)

def assign_action(row):
    if row["Stockout_Risk"] == 1 and row["Overstock_Risk"] == 1:
        return "Watch/Volatile"
    elif row["Stockout_Risk"] == 1:
        return "Reorder Now"
    elif row["Overstock_Risk"] == 1:
        return "Markdown/Clear"
    else:
        return "Healthy"

latest_snapshot["Recommended_Action"] = latest_snapshot.apply(assign_action, axis=1)

latest_snapshot["Revenue_at_Stake"] = latest_snapshot["Forecast_Next_4wk"] * latest_snapshot["Price"]

risk_output = latest_snapshot[[
    "Product ID", "Category", "Inventory_Level", "Forecast_Next_4wk",
    "Stockout_Risk", "Overstock_Risk", "Recommended_Action", "Revenue_at_Stake"
]].sort_values("Revenue_at_Stake", ascending=False)

risk_output

,Product ID,Category,Inventory_Level,Forecast_Next_4wk,Stockout_Risk,Overstock_Risk,Recommended_Action,Revenue_at_Stake
1154,P0014,Furniture,3733,4074.887451,0,0,Healthy,519397.379189
1151,P0011,Clothing,3023,4941.250000,0,0,Healthy,497900.115000
1158,P0009,Clothing,2058,5107.201172,1,0,Reorder Now,426313.403420
1159,P0020,Clothing,2267,4310.964355,1,0,Reorder Now,410757.305718
1156,P0016,Electronics,2157,5153.963379,1,0,Reorder Now,387541.968350
1152,P0012,Electronics,3404,3632.162109,0,0,Healthy,361835.989336
1141,P0018,Furniture,2075,4096.135742,1,0,Reorder Now,334789.462616
1149,P0019,Groceries,1850,3479.140625,1,0,Reorder Now,319051.111875
1150,P0008,Furniture,1531,4454.799316,1,0,Reorder Now,317119.344138
1144,P0003,Furniture,2985,3954.295654,0,0,Healthy,317011.928309


In [12]:
risk_output.to_csv("../data/processed/latest_risk_scores.csv", index=False)
print("Saved:", risk_output.shape)

Saved: (20, 8)


In [13]:
raw_df = pd.read_csv("../data/raw/demand_forecasting.csv")

raw_df.isna().sum()

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Price                 0
Discount              0
Weather Condition     0
Promotion             0
Competitor Pricing    0
Seasonality           0
Epidemic              0
Demand                0
dtype: int64

In [14]:

weekly[["Product ID", "Week", "Demand", "Category", "Price"]].to_csv(
    "../data/processed/weekly_demand.csv", index=False
)
print("Saved weekly_demand.csv:", weekly.shape)

Saved weekly_demand.csv: (2200, 20)


In [15]:

all_preds_df.to_csv("../data/processed/backtest_predictions.csv", index=False)
print("Saved backtest_predictions.csv:", all_preds_df.shape)

Saved backtest_predictions.csv: (880, 6)


In [17]:
raw_df = pd.read_csv("../data/raw/demand_forecasting.csv")

print("missing values")
print(raw_df.isna().sum())

missing values
Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Price                 0
Discount              0
Weather Condition     0
Promotion             0
Competitor Pricing    0
Seasonality           0
Epidemic              0
Demand                0
dtype: int64


In [18]:
print("\nduplicates")
print(raw_df.duplicated().sum())



duplicates
0


In [19]:
print("\ndemand by category")
print(raw_df.groupby("Category")["Demand"].agg(["mean", "sum"]).sort_values("sum", ascending=False))


demand by category
                   mean      sum
Category                        
Groceries    120.976447  3677684
Clothing     112.619737  1369456
Furniture     73.581140  1006590
Toys          92.606955   985338
Electronics   97.482018   889036


In [20]:
print("\ntop 5 skus")
print(raw_df.groupby("Product ID")["Demand"].sum().sort_values(ascending=False).head(5))

print("\nbottom 5 skus")
print(raw_df.groupby("Product ID")["Demand"].sum().sort_values(ascending=True).head(5))


top 5 skus
Product ID
P0009    450324
P0013    440895
P0004    438505
P0007    437089
P0002    425853
Name: Demand, dtype: int64

bottom 5 skus
Product ID
P0020    320847
P0012    347215
P0017    350620
P0014    364335
P0019    371954
Name: Demand, dtype: int64


In [21]:
print("\ndemand by season")
print(raw_df.groupby("Seasonality")["Demand"].mean().sort_values(ascending=False))

print("\npromotion impact")
print(raw_df.groupby("Promotion")["Demand"].mean())


demand by season
Seasonality
Summer    112.855380
Autumn    103.422967
Winter    103.406190
Spring     97.703098
Name: Demand, dtype: float64

promotion impact
Promotion
0     95.026843
1    123.269400
Name: Demand, dtype: float64
